# T5: チームメイト比較分析

全10チーム × 3GP (R01 Australia / R02 China / R03 Japan) のチームメイトペアを比較する。

## 比較指標
1. **PaceDelta_sec**: 燃料補正済みペース中央値差（秒）
2. **S1/S2/S3 Delta**: セクター別タイム中央値差（秒）
3. **SpeedDelta**: スピードトラップ(SpeedST)中央値差（km/h）
4. **PositionDelta**: レース最終順位差
5. **TyreDegDelta**: タイヤ劣化差（後半50%ペース - 前半50%ペース の差）

**Delta = Driver1 - Driver2** (負 = Driver1 が速い)


In [ ]:
import matplotlib
matplotlib.use('Agg')  # GUIなし環境でも動作

import csv, os, math, statistics
from collections import defaultdict
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.font_manager as fm
import numpy as np

# 日本語フォント設定
def _setup_jp_font():
    candidates = ['Hiragino Sans', 'Hiragino Kaku Gothic ProN', 'AppleGothic']
    available = {f.name for f in fm.fontManager.ttflist}
    for c in candidates:
        if c in available:
            matplotlib.rcParams['font.family'] = c
            print(f'日本語フォント: {c}')
            return c
    print('警告: 日本語フォントなし')
    return 'DejaVu Sans'

_JP_FONT = _setup_jp_font()

# パス設定
BASE_DIR   = os.path.abspath('.')
ROOT_DIR   = os.path.dirname(BASE_DIR)
OUTPUT_DIR = os.path.join(BASE_DIR, 'output')
os.makedirs(OUTPUT_DIR, exist_ok=True)

LONGRUNS_CSV = os.path.join(OUTPUT_DIR, 'clean_longruns.csv')
FUEL_CSV     = os.path.join(OUTPUT_DIR, 'fuel_corrected_pace.csv')

GP_META = {
    'R01': {'label': 'Australia',
            'results': os.path.join(ROOT_DIR, 'data/2026_R01_Australia/export/race_results.csv'),
            'laps':    os.path.join(ROOT_DIR, 'data/2026_R01_Australia/export/race_laps.csv')},
    'R02': {'label': 'China',
            'results': os.path.join(ROOT_DIR, 'data/2026_R02_China/export/race_results.csv'),
            'laps':    os.path.join(ROOT_DIR, 'data/2026_R02_China/export/race_laps.csv')},
    'R03': {'label': 'Japan',
            'results': os.path.join(ROOT_DIR, 'data/2026_R03_Japan/export/race_results.csv'),
            'laps':    os.path.join(ROOT_DIR, 'data/2026_R03_Japan/export/race_laps.csv')},
}

TEAMMATE_PAIRS = {
    'McLaren':         ('NOR', 'PIA'),
    'Ferrari':         ('LEC', 'HAM'),
    'Red Bull Racing': ('VER', 'HAD'),
    'Mercedes':        ('RUS', 'ANT'),
    'Aston Martin':    ('ALO', 'STR'),
    'Williams':        ('ALB', 'SAI'),
    'Racing Bulls':    ('LAW', 'LIN'),
    'Alpine':          ('GAS', 'COL'),
    'Haas F1 Team':    ('OCO', 'BEA'),
    'Audi':            ('HUL', 'BOR'),
}

def safe_median(vals):
    clean = [v for v in vals if v is not None and not math.isnan(v)]
    return statistics.median(clean) if clean else None

def to_float(s):
    try:
        v = float(s)
        return None if math.isnan(v) else v
    except (TypeError, ValueError):
        return None

print('セットアップ完了')


In [ ]:
# データ読み込み

def load_fuel_corrected():
    result = {}
    with open(FUEL_CSV, newline='', encoding='utf-8') as f:
        for row in csv.DictReader(f):
            gp_key = row['GP'].split('_')[0]
            pace = to_float(row['FuelCorrectedMedianPace'])
            result[(gp_key, row['Driver'])] = pace
    print(f'[燃料補正ペース] {len(result)} レコード')
    return result

def load_longruns():
    data = defaultdict(lambda: {'s1': [], 's2': [], 's3': [], 'laps': []})
    with open(LONGRUNS_CSV, newline='', encoding='utf-8') as f:
        for row in csv.DictReader(f):
            gp_key = row['GP'].split('_')[0]
            key = (gp_key, row['Driver'])
            for col, lst in [('Sector1Time_sec','s1'), ('Sector2Time_sec','s2'), ('Sector3Time_sec','s3')]:
                v = to_float(row[col])
                if v is not None:
                    data[key][lst].append(v)
            lt = to_float(row['LapTime_sec'])
            tl = to_float(row['TyreLife'])
            if lt is not None and tl is not None:
                data[key]['laps'].append((tl, lt))
    print(f'[ロングラン] {len(data)} ドライバー×GPキー')
    return data

def load_speed_st():
    result = {}
    for gp_key, meta in GP_META.items():
        if not os.path.exists(meta['laps']):
            continue
        driver_speeds = defaultdict(list)
        with open(meta['laps'], newline='', encoding='utf-8') as f:
            for row in csv.DictReader(f):
                spd = to_float(row.get('SpeedST'))
                if spd is not None and spd > 0:
                    driver_speeds[row['Driver']].append(spd)
        for drv, speeds in driver_speeds.items():
            result[(gp_key, drv)] = safe_median(speeds)
    print(f'[SpeedST] {len(result)} レコード')
    return result

def load_race_positions():
    result = {}
    for gp_key, meta in GP_META.items():
        if not os.path.exists(meta['results']):
            continue
        with open(meta['results'], newline='', encoding='utf-8') as f:
            for row in csv.DictReader(f):
                result[(gp_key, row['Abbreviation'])] = to_float(row.get('Position'))
    print(f'[レース順位] {len(result)} レコード')
    return result

fuel_pace = load_fuel_corrected()
longruns  = load_longruns()
speed_st  = load_speed_st()
race_pos  = load_race_positions()


In [ ]:
# チームメイトデルタ計算

def tyre_deg_delta(laps):
    """後半ペース中央値 - 前半ペース中央値（正=劣化、負=燃料効果優勢）"""
    if len(laps) < 4:
        return None
    sorted_laps = sorted(laps, key=lambda x: x[0])
    mid = len(sorted_laps) // 2
    first_half_times  = [lt for _, lt in sorted_laps[:mid]]
    second_half_times = [lt for _, lt in sorted_laps[mid:]]
    m1 = safe_median(first_half_times)
    m2 = safe_median(second_half_times)
    return (m2 - m1) if (m1 is not None and m2 is not None) else None

rows = []
for team, (d1, d2) in TEAMMATE_PAIRS.items():
    for gp in ['R01', 'R02', 'R03']:
        gp_label = GP_META[gp]['label']

        # 燃料補正済みペース差
        p1, p2 = fuel_pace.get((gp, d1)), fuel_pace.get((gp, d2))
        pace_delta = (p1 - p2) if (p1 is not None and p2 is not None) else None

        # セクタータイム差
        lr1 = longruns.get((gp, d1), {'s1':[],'s2':[],'s3':[],'laps':[]})
        lr2 = longruns.get((gp, d2), {'s1':[],'s2':[],'s3':[],'laps':[]})

        def sec_d(s):
            m1, m2 = safe_median(lr1[s]), safe_median(lr2[s])
            return (m1 - m2) if (m1 is not None and m2 is not None) else None

        # SpeedST 差
        sp1, sp2 = speed_st.get((gp, d1)), speed_st.get((gp, d2))
        speed_delta = (sp1 - sp2) if (sp1 is not None and sp2 is not None) else None

        # 最終順位差
        pos1, pos2 = race_pos.get((gp, d1)), race_pos.get((gp, d2))
        pos_delta = (pos1 - pos2) if (pos1 is not None and pos2 is not None) else None

        # タイヤ劣化差
        deg1 = tyre_deg_delta(lr1['laps'])
        deg2 = tyre_deg_delta(lr2['laps'])
        tyre_deg = (deg1 - deg2) if (deg1 is not None and deg2 is not None) else None

        rows.append({
            'GP': gp_label, 'Team': team, 'Driver1': d1, 'Driver2': d2,
            'PaceDelta_sec': round(pace_delta, 4) if pace_delta is not None else None,
            'S1Delta':       round(sec_d('s1'), 4) if sec_d('s1') is not None else None,
            'S2Delta':       round(sec_d('s2'), 4) if sec_d('s2') is not None else None,
            'S3Delta':       round(sec_d('s3'), 4) if sec_d('s3') is not None else None,
            'SpeedDelta':    round(speed_delta, 2) if speed_delta is not None else None,
            'PositionDelta': int(pos_delta) if pos_delta is not None else None,
            'TyreDegDelta':  round(tyre_deg, 4) if tyre_deg is not None else None,
        })

        status = f"{pace_delta:+.3f}s" if pace_delta is not None else "NaN"
        print(f"  {gp} {team:20s} {d1} vs {d2}: PaceDelta={status}")

print(f"\n合計 {len(rows)} 行")


In [ ]:
# CSV 出力
import csv as csv_module

fieldnames = ['GP', 'Team', 'Driver1', 'Driver2',
              'PaceDelta_sec', 'S1Delta', 'S2Delta', 'S3Delta',
              'SpeedDelta', 'PositionDelta', 'TyreDegDelta']

csv_path = os.path.join(OUTPUT_DIR, 'teammate_comparison.csv')
with open(csv_path, 'w', newline='', encoding='utf-8') as f:
    w = csv_module.DictWriter(f, fieldnames=fieldnames, extrasaction='ignore')
    w.writeheader()
    for row in rows:
        w.writerow({k: ('' if v is None else v) for k, v in row.items()})

print(f'保存: {csv_path} ({len(rows)} 行)')
print()
print(f"{'GP':12} {'Team':20} {'D1':5} {'D2':5} {'PaceDelta':>10} {'PosDelta':>9}")
print('-' * 70)
for r in rows:
    pd_str  = f"{r['PaceDelta_sec']:+.3f}" if r['PaceDelta_sec'] is not None else '  NaN  '
    pos_str = str(r['PositionDelta']) if r['PositionDelta'] is not None else 'NaN'
    print(f"{r['GP']:12} {r['Team']:20} {r['Driver1']:5} {r['Driver2']:5} {pd_str:>10} {pos_str:>9}")


In [ ]:
# ヒートマップ描画

gp_order   = ['Australia', 'China', 'Japan']
team_order = list(TEAMMATE_PAIRS.keys())
pair_labels = {t: f'{d1} vs {d2}' for t, (d1, d2) in TEAMMATE_PAIRS.items()}

matrix = np.full((len(team_order), len(gp_order)), np.nan)
for row in rows:
    ti = team_order.index(row['Team'])
    gi = gp_order.index(row['GP'])
    if row['PaceDelta_sec'] is not None:
        matrix[ti, gi] = float(row['PaceDelta_sec'])

abs_max = np.nanmax(np.abs(matrix)) if not np.all(np.isnan(matrix)) else 1.0
norm = mcolors.TwoSlopeNorm(vmin=-abs_max, vcenter=0, vmax=abs_max)

fig, ax = plt.subplots(figsize=(10, 8))
fig.patch.set_facecolor('#1a1a2e')
ax.set_facecolor('#1a1a2e')

im = ax.imshow(matrix, cmap='RdBu_r', norm=norm, aspect='auto')

cbar = fig.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label('Pace Delta [sec]\n(negative = Driver1 faster)', color='white', fontsize=10)
cbar.ax.yaxis.set_tick_params(color='white')
plt.setp(cbar.ax.yaxis.get_ticklabels(), color='white')

ax.set_xticks(range(len(gp_order)))
ax.set_xticklabels(gp_order, color='white', fontsize=11)
ax.set_yticks(range(len(team_order)))
ax.set_yticklabels(
    [f'{t}\n({pair_labels[t]})' for t in team_order],
    color='white', fontsize=9
)
ax.tick_params(colors='white')

for ti in range(len(team_order)):
    for gi in range(len(gp_order)):
        val = matrix[ti, gi]
        if not np.isnan(val):
            tc = 'white' if abs(val) > abs_max * 0.5 else '#cccccc'
            ax.text(gi, ti, f'{val:+.3f}', ha='center', va='center',
                    color=tc, fontsize=9, fontweight='bold')
        else:
            ax.text(gi, ti, 'N/A', ha='center', va='center',
                    color='#666688', fontsize=8)

ax.set_title('Teammate Pace Delta Heatmap (2026 R01-R03)\nDriver1 - Driver2 [sec, fuel-corrected]',
             color='white', fontsize=13, pad=15)
ax.spines[:].set_color('#333355')

plt.tight_layout()
heatmap_path = os.path.join(OUTPUT_DIR, 'teammate_heatmap.png')
plt.savefig(heatmap_path, dpi=150, bbox_inches='tight', facecolor='#1a1a2e')
plt.show()
print(f'保存: {heatmap_path}')


## 結果の解釈

- **青セル（負）**: Driver1 が Driver2 より速い
- **赤セル（正）**: Driver2 が Driver1 より速い
- **N/A**: 一方または両方のドライバーにロングランデータなし（DNF等）

### 注意事項
- 燃料補正は均一モデル（+0.06秒/ラップ）。チーム別の実燃費差は非公開のため近似値
- McLaren R01/R02 に N/A → どちらかのドライバーが5周未満のロングランしか記録していない可能性あり
- TyreDegDelta は燃料効果を含むため、単純なタイヤ劣化比較ではない（T3参照）
